# Session 18 · Random Forest II — Choosing a Model

We close the module (and cross the halfway point of the course) by doing the real job:
put three models on one problem, measure them the same way, and **choose**.

Three contenders on the fraud data: a **decision tree** (S14–16), a **random forest**
(S17), and **logistic regression** (Module 3). One split, one metrics table.

> ✏️ = your cell. Gaps never block the run.

## Step 1 · Setup — one problem, one yardstick

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

fraud = pd.read_csv("../../../datasets/secondary/fraud_transactions.csv")
features = ["amount","hour_of_day","is_online","distance_from_home_km","transactions_last_24h"]
X, y = fraud[features], fraud['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('test transactions:', len(y_test), ' fraud:', int(y_test.sum()))

## Step 2 · Run the workflow loop THREE times

Same loop (fit → predict → evaluate), once per model. This *is* the workflow mantra,
run three times and compared.

In [ ]:
# Three contenders. Only logistic needs scaling (trees are scale-invariant),
# so we wrap it in a pipeline that scales first.
tree     = DecisionTreeClassifier(random_state=0)
forest   = RandomForestClassifier(n_estimators=100, random_state=0)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

models = {'decision tree': tree, 'random forest': forest, 'logistic regression': logistic}
for m in models.values():
    m.fit(X_train, y_train)
print('all three models fitted on the same split.')

## Step 3 · Build the comparison table

Accuracy, precision, recall for all three — side by side, on the same held-out test set.

In [ ]:
rows = []
for name, m in models.items():
    p = m.predict(X_test)
    rows.append({'model': name,
                 'accuracy':  round(accuracy_score(y_test, p), 3),
                 'precision': round(precision_score(y_test, p), 3),
                 'recall':    round(recall_score(y_test, p), 3)})
table = pd.DataFrame(rows).set_index('model')
print(table.to_string())

## Step 4 · The surprise — read the table across

For each metric, which model is highest? Before you scroll: you might expect the fancy
**forest** to win. Let the numbers speak.

In [ ]:
print('Winner of each metric:')
for col in ['accuracy', 'precision', 'recall']:
    winner = table[col].idxmax()
    print(f'   {col:10s}: {winner}  ({table[col].max()})')
print()
print('The SIMPLE model — logistic regression — wins every metric.')
print('Two sessions building forests, and the linear model from Module 3 takes it.')
print('The lesson is not "forests are bad" — it is: you had to MEASURE. On this')
print('fairly linear fraud signal, a linear model fits best. On another problem the')
print('forest could win. No single algorithm is best everywhere.')

## Step 5 · The second axis — interpretability

A metrics table can't hold everything. **Can you explain a decision to the person it
affects?** Read each model's transparency.

In [ ]:
# decision tree: read a shallow version aloud as rules
shallow = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
print('DECISION TREE — reads as if-then rules:')
print(export_text(shallow, feature_names=features))

# logistic: a signed weight per feature (biggest = strongest driver)
coef = logistic.named_steps['logisticregression'].coef_[0]
weights = pd.Series(coef, index=features).sort_values(key=abs, ascending=False)
print('LOGISTIC REGRESSION — signed weight per feature (scaled):')
for f, w in weights.items():
    print(f'   {f:26s} {w:+.2f}  ({"raises" if w > 0 else "lowers"} fraud probability)')
print()
print('RANDOM FOREST — a black box: 100 voting trees. It can rank feature importance,')
print('but it cannot give ONE clean reason for ONE customer''s declined transaction.')

## Step 6 · ✏️ Your model recommendation (the module's synthesis)

You advise a bank's **fraud team**. Write a **6–7 sentence recommendation** that:
1. names the model you'd ship and **justifies it with specific numbers** from the table;
2. weighs **interpretability** — when the bank declines a transaction, the customer is
   owed a reason. Which models can give one? Which can't?
3. names **who is owed an explanation** and whether your choice honours that;
4. acknowledges that a **different problem** (e.g. with strong non-linear patterns)
   could change the answer.

*(Write your recommendation here, replacing this line. There is no single correct pick —
you're graded on numbers + interpretability + stakeholders, not on which model you name.)*

## Wrap-up — and the end of Module 4

- Choosing a model is a **judgement made by measuring** several on one yardstick — not by
  assuming the most complex wins. Here the **simple** model won; we only know because we checked.
- Weigh a second axis the table can't show: **interpretability** and **who is owed an explanation**.
- The best model **fits the problem and its people** — re-run the comparison for every new problem.

**The module in one line:** trees are readable → deep trees overfit → forests fix that at a
cost → and the model you *ship* is the one the numbers and the stakeholders justify.

**Next (Module 5):** we leave labels behind for **unsupervised learning** — finding structure
with no answer key. Overfitting returns there as *over-segmentation*.